In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import pandas as pd
from glob import glob
import random
import matplotlib.cm as cm
from input.init import model
from input.visualization import extract_min_max_price_metric, replace_nans_and_zeros,_create_heatmap_gamma_only,create_comparative_heatmaps_gl,extract_metric_data_gl, extract_metric_data

: 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob

# ———————————————
# helper as before
def load_gamma_metric(experiment_dir, metric_name):
    pattern = "gamma_*_lambda_*"
    run_dirs = glob(os.path.join(experiment_dir, pattern))

    gammas, values = [], []
    for run in run_dirs:
        base = os.path.basename(run)
        try:
            γ = float(base.split('gamma_')[1].split('_lambda_')[0])
        except:
            continue

        stats_file = os.path.join(run, "cycle_statistics.csv")
        if not os.path.exists(stats_file):
            continue
        df = pd.read_csv(stats_file)
        if metric_name == "Cycle Length":
            val = float(df["mean_cycle_length"].iloc[0])
        elif metric_name == "Surplus":
            val = float(df["mean_consumer_surplus"].iloc[0])
        else:
            key = metric_name.lower().replace(" ", "_")
            cols = [c for c in df.columns if c.startswith(f"mean_{key}_p")]
            val = df[cols].mean(axis=1).iloc[0] if cols else np.nan

        gammas.append(γ)
        values.append(val)

    if not gammas:
        raise ValueError(f"No data in {experiment_dir} for '{metric_name}'")

    df2 = pd.DataFrame({"gamma": gammas, "value": values}).dropna()
    grouped = df2.groupby("gamma")["value"].agg(["mean", "std"]).reset_index()
    return grouped["gamma"].values, grouped["mean"].values, grouped["std"].values



# ———————————————
# NEW: loader for gamma‑only runs
def load_gamma_only_metric(experiment_dir, metric_name):
    pattern = "gamma_*"
    run_dirs = glob(os.path.join(experiment_dir, pattern))

    gammas, means, stds = [], [], []
    for run in run_dirs:
        base = os.path.basename(run)
        try:
            γ = float(base.split("gamma_")[1])
        except:
            continue

        stats_file = os.path.join(run, "cycle_statistics.csv")
        if not os.path.exists(stats_file):
            continue

        df = pd.read_csv(stats_file)
        # pick your metric exactly as in load_gamma_metric,
        # but also grab std from 'std_<metric>_p*' columns:
        key = metric_name.lower().replace(" ", "_")
        mean_cols = [c for c in df.columns if c.startswith(f"mean_{key}_p")]
        std_cols  = [c for c in df.columns if c.startswith(f"std_{key}_p")]

        if not mean_cols or not std_cols:
            continue

        mval = df[mean_cols].mean(axis=1).iloc[0]
        sval = df[std_cols].mean(axis=1).iloc[0]

        gammas.append(γ)
        means.append(mval)
        stds.append(sval)

    if not gammas:
        raise ValueError(f"No γ-only data in {experiment_dir} for '{metric_name}'")

    # sort by γ
    idx = np.argsort(gammas)
    return (np.array(gammas)[idx],
            np.array(means)[idx],
            np.array(stds)[idx])

# ———————————————
# paths
folder   = "../Results_sockeye/experiments"
base     = "reference_two_gamma_lambda_sockeye/gamma_lambda_c"
base2 = "reference_two_gamma_lambda_sockeye/gamma"

exp_dirs = {
    ('reference',        True) : os.path.join(folder, f"{base}_reference_True"),
    ('reference',        False): os.path.join(folder, f"{base}_reference_False"),
    ('misspecification', True) : os.path.join(folder, f"{base}_misspecification_True"),
    ('misspecification', False): os.path.join(folder, f"{base}_misspecification_False"),
    #  add the q‑learning folders:
    ('qlr_reference',    True) : os.path.join(folder, f"{base2}_only_qlr_reference_True"),
    ('qlr_reference',    False): os.path.join(folder, f"{base2}_only_qlr_reference_False"),
}

out_dir = os.path.join(folder, base, "Gamma_Figures")
os.makedirs(out_dir, exist_ok=True)

metrics = ["Price", "Profit", "Price Gain", "Profit Gain"]

# ———————————————
# plotting
ref_color = 'tab:blue'
mis_color = 'tab:orange'
qlr_color= 'tab:purple'

for metric_name in metrics:
    fig, ax = plt.subplots(figsize=(10,6))

    # first the four grid-search curves
    for demand, cr in [('reference', True),
                       ('reference', False),
                       ('misspecification', True),
                       ('misspecification', False)]:
        γ, μ, σ = load_gamma_metric(exp_dirs[(demand, cr)], metric_name)
        color     = ref_color if demand=='reference' else mis_color
        linestyle = '-'       if cr       else '--'
        label     = f"{demand.capitalize()} (CR={cr})"

        ax.plot(γ, μ, color=color, linestyle=linestyle,
                marker='o', linewidth=2, label=label)
        ax.fill_between(γ, μ-σ, μ+σ, color=color, alpha=0.15)

    # now overlay the two q‑learning curves
    for cr in [True, False]:
        γ, μ, σ = load_gamma_only_metric(exp_dirs[('qlr_reference', cr)], metric_name)
        linestyle = '-' if cr else '--'
        label     = f"Q-learning Ref (CR={cr})"

        ax.plot(γ, μ, color=qlr_color, linestyle=linestyle,
                marker='D', linewidth=2, label=label)
        ax.fill_between(γ, μ-σ, μ+σ, color=qlr_color, alpha=0.15)

    # finalize
    ax.set_xlabel(r"$\gamma$", fontsize=12)
    ax.set_ylabel(metric_name, fontsize=12)
    ax.set_title(f"{metric_name} vs γ ", fontsize=14)
    ax.grid(True, linestyle="--", alpha=0.5)

    ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout(rect=[0,0,0.85,1])

    fname = metric_name.lower().replace(" ", "_") + "_gamma_all6.png"
    fig.savefig(os.path.join(out_dir, fname), dpi=300)
    plt.close(fig)
    print(f"Saved → {fname}")

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np # Assuming load_gamma_metric might return numpy arrays or pandas Series

# --- Assume these are defined elsewhere in your full script ---
# Make sure these variables are defined before this plotting block
# out_dir = "path/to/your/output_directory"
# metrics = ["Price", "Profit", "Price Gain", "Profit Gain"] # CSV column names
# exp_dirs = { # Your full exp_dirs dictionary with paths to data
#    ('reference', True): "path/to/ref_true_data",
#    ('misspecification', True): "path/to/mis_true_data",
#    # ... other entries if needed for Q-learning later
# }
# def load_gamma_metric(path, metric_name):
#     # ... your actual data loading logic ...
#     # Should return: gamma_series, mean_series, std_series
#     # Example placeholder:
#     # gamma_vals = pd.Series(np.linspace(0, 3, 31)) # 31 points for 0 to 3 with step 0.1
#     # if "Price" in metric_name:
#     #     base_val = 1.65 if "miss" in path else 1.625
#     #     means = base_val - gamma_vals * 0.05
#     # elif "Profit" in metric_name:
#     #     base_val = 0.285 if "miss" in path else 0.275
#     #     means = base_val - gamma_vals * 0.02
#     # elif "Gain" in metric_name:
#     #     base_val = 0.42 if "miss" in path else 0.38
#     #     means = base_val + gamma_vals * 0.12
#     # stds = pd.Series(np.random.rand(31) * 0.03 + 0.02) # Wider error bands for misspec
#     # return gamma_vals, means, stds
#     pass # Replace with actual implementation

# ref_color = 'tab:blue'    # Color for 'reference' case
# mis_color = 'tab:orange'  # Color for 'misspecification' case
# qlr_color = 'tab:green' # If you uncomment Q-learning part

# --- End of assumed definitions ---


# ------------------------------------------------------------------
# 2×2 “quad” figure: Misspecification vs. Reference
# ------------------------------------------------------------------

# MODIFICATION: Enhanced plt.rcParams (consistent with previous figures)
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14, # Slightly larger legend font
    'figure.titlesize': 18,
    'font.family': 'serif',
    'mathtext.fontset': 'dejavuserif',
})

# MODIFICATION: Configuration for human-readable metric names for titles
# (Assuming 'metrics' list contains the keys used here)
metrics_display_config = {
    "Price": "Price", # Displayed title for the "Price" metric
    "Profit": "Profit",
    "Price Gain": "Price Gain",
    "Profit Gain": "Profit Gain"
}
# Ensure 'metrics' is a list of keys present in metrics_display_config, e.g.,
# metrics = ["Price", "Profit", "Price Gain", "Profit Gain"]


# MODIFICATION: Increased figsize for better spacing and legend
fig_q, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
axes_flat = axes.flatten() # Use axes_flat for direct indexing

# We’ll collect handles and labels from the first subplot for the shared legend
# This ensures the legend order matches the plotting order if consistent
plot_handles_for_legend = []
plot_labels_for_legend = []
legend_elements_collected = False # Flag to collect legend items only once

# Iterate using enumerate for index, to apply x-label to bottom row
for idx, metric_csv_col_name in enumerate(metrics):
    ax = axes_flat[idx]
    # Use the display name from config, or fallback to the original metric name
    human_readable_title = metrics_display_config.get(metric_csv_col_name, metric_csv_col_name)

    first_gamma_series_for_xlim = None # To set x-limits based on actual data

    # ---------- Plotting Reference and Misspecification curves ----------
    # Define the order and properties for plotting
    plot_configs = [
        {'key': ('reference', True), 'color': ref_color, 'label_base': "Correctly Specified"},
        {'key': ('misspecification', True), 'color': mis_color, 'label_base': "Misspecification"}
    ]

    for config in plot_configs:
        demand_type, cr_flag = config['key']
        
        if config['key'] not in exp_dirs:
            print(f"Warning: Key {config['key']} not found in exp_dirs. Skipping plot for {config['label_base']}.")
            continue
            
        # Load data: gamma_values, mean_values, std_dev_values
        γ, μ, σ = load_gamma_metric(exp_dirs[config['key']], metric_csv_col_name)
        
        if γ is None or (hasattr(γ, 'empty') and γ.empty): # Check for empty or None data
            print(f"Warning: No data loaded for {config['label_base']} (CR={cr_flag}), metric {metric_csv_col_name}. Skipping plot.")
            continue
        
        if first_gamma_series_for_xlim is None: # Store the first valid gamma series
            first_gamma_series_for_xlim = γ

        # MODIFICATION: Consistent label format, CR=True is implied by the legend title
        label_text = config['label_base'] # Simplified label, as CR=True is constant here

        line, = ax.plot(γ, μ, color=config['color'], linestyle='-', # Solid line for both
                        marker='o', linewidth=2, markersize=6, label=label_text)
        ax.fill_between(γ, μ-σ, μ+σ, color=config['color'], alpha=0.2) # Slightly more visible error band

        # Collect legend items only from the first subplot and if not already collected
        if not legend_elements_collected:
            plot_handles_for_legend.append(line)
            plot_labels_for_legend.append(label_text)

    if not legend_elements_collected and plot_handles_for_legend:
        legend_elements_collected = True # Mark as collected after processing all lines for the first subplot

    # # ---------- Q‑learning curves (user's original commented out code - can be integrated similarly if needed) ----------
    # # ...

    # MODIFICATION: Enhanced axis cosmetics
    ax.set_title(human_readable_title)
    ax.grid(True, linestyle=':', linewidth=0.7, alpha=0.6) # Lighter, dotted grid
    ax.set_ylabel(human_readable_title) # Using descriptive title for y-label
    ax.tick_params(direction='in', top=True, right=True) # Inward ticks for a cleaner look


    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

    # X-axis label for bottom row only
    if idx >= 2 : # Plots with index 2 and 3 are on the bottom row of a 2x2 grid
        ax.set_xlabel(r"$\gamma$ (Reference Dependence Strength)")
    # else: # Optional: remove x-tick labels from top plots when sharex=True
    #     ax.tick_params(labelbottom=False)

    # Set xlim based on actual data if available, otherwise default
    if first_gamma_series_for_xlim is not None and first_gamma_series_for_xlim.size > 0:
         ax.set_xlim(left=first_gamma_series_for_xlim.min(), right=first_gamma_series_for_xlim.max())
    else:
        ax.set_xlim(0, 3) # Default x-axis limits if no data


# # Try reducing the 'y' value for suptitle slightly if it's too high
# fig_q.suptitle("Impact of Model Misspecification on Market Outcomes (CR=True)", y=0.95, fontsize=plt.rcParams['figure.titlesize']) # REDUCED y from 1.02

# ---------------- Shared Legend (Top of the figure) & Layout ----------------
fig_q.legend(plot_handles_for_legend,
             plot_labels_for_legend,
             loc='upper center',
             bbox_to_anchor=(0.4, 1.02),  # Centered just above the plots
             ncol=len(plot_handles_for_legend),
             frameon=False)

# MODIFICATION: Adjusted tight_layout for suptitle and legend
fig_q.tight_layout(rect=[0, 0, 0.85, 1])

# MODIFICATION: More descriptive filename
quad_fname = "gamma_misspecification_vs_reference_quad.png"
fig_q.savefig(os.path.join(out_dir, quad_fname), dpi=300,
              bbox_inches='tight')

print(f"Figure saved to {os.path.join(out_dir, quad_fname)}")
# plt.show() # Uncomment to display the plot